# SAM 3 + YOLO11m-seg + Active Learning (Pre-label Tập Dữ Liệu Mới)

**Thiết lập trên Kaggle trước khi chạy:**
1. **Accelerator:** `GPU T4 x2` hoặc `GPU P100` (16 GB VRAM)
2. **Internet:** `ON`
3. **Input:** Bấm **Add Input** gắn `Sam3_weight` **VÀ** Dataset chứa bộ ảnh mới của bạn.

In [ ]:
!nvidia-smi
!free -h
%cd /kaggle/working
!rm -rf hackathon_vinai
!git clone https://github.com/bayvai20kg/hackathon_vinai.git
%cd /kaggle/working/hackathon_vinai
!pip install -q "ultralytics>=8.4.210" cvat-sdk huggingface_hub psutil PyYAML tqdm
import torch
print("PyTorch:", torch.__version__, "CUDA:", torch.cuda.is_available(), torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU")

In [ ]:
# Tự động nối sam3.pt từ /kaggle/input/datasets/khangvu0510/sam3-weight vào weights/sam3.pt
!find /kaggle/input -maxdepth 4
!mkdir -p weights
!rm -f weights/sam3.pt
!SAM3_SRC=$(find /kaggle/input/datasets/khangvu0510/sam3-weight -type f 2>/dev/null | head -n 1); if [ -z "$SAM3_SRC" ]; then SAM3_SRC=$(find /kaggle/input -name "sam3.pt" -type f | head -n 1); fi; if [ -n "$SAM3_SRC" ]; then ln -sf "$SAM3_SRC" weights/sam3.pt; fi
!ls -lhL weights/sam3.pt 2>/dev/null || echo "Chưa tìm thấy sam3.pt -> sẽ dùng SAM 2.1 fallback"

## Bước 1: Cấu hình Pre-label tập dữ liệu mới
- Đã đặt sẵn **`MODE = "prelabel"`** để chạy thẳng **YOLO11m-seg + YOLOE-11l + SAM 3** trên toàn bộ ảnh mới (kèm thanh tiến trình `tqdm`).
- Nếu bạn để `NEW_IMAGES_DIR = "auto"`, code sẽ **tự tìm thư mục ảnh** trong `/kaggle/input` (bỏ qua `sam3-weight`). Hoặc bạn điền thẳng đường dẫn cụ thể (ví dụ `/kaggle/input/my-ds/images`).

In [ ]:
# === CẤU HÌNH PRE-LABEL DATASET MỚI ===
MODE = "prelabel"  # Chạy pre-label thẳng luôn: "prelabel" | "train_cvat" | "train_yaml" | "demo_50"

NEW_IMAGES_DIR = "/kaggle/input/datasets/khangvu0510/vietnam-extra/extra_train/extra_train/images"  # Đường dẫn thư mục chứa ảnh mới
NEW_LABELS_OR_DATA = ""  # Để "" nếu ảnh chưa có nhãn; nếu có nhãn cũ/CVAT thì điền đường dẫn

WORK_DIR = "pipeline_work_new"   # Tách riêng kết quả của dataset mới
BATCH_NAME = "new_ds_prelabel"
RUN_NAME = "new_ds_yolo11m"

NUM_IMAGES = 0    # 0 = chạy toàn bộ ảnh trong dataset mới
EPOCHS = 40
BATCH = 8
IMGSZ = 960
WEIGHTS_INIT = "yolo11m-seg.pt"

In [ ]:
# === CHẠY PIPELINE (hiển thị thanh tiến trình trực tiếp) ===
import shlex
from pathlib import Path
from IPython import get_ipython

def _find_kaggle_image_root():
    exts = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}
    candidates = []
    input_root = Path("/kaggle/input")
    if not input_root.exists():
        return None
    for p in sorted(input_root.rglob("*")):
        if p.is_file() and p.suffix.lower() in exts and "sam3-weight" not in str(p):
            candidates.append(p.parent)
            if len(candidates) >= 200:
                break
    if not candidates:
        return None
    common = candidates[0]
    for c in candidates[1:]:
        while common != common.parent and common not in c.parents and common != c:
            common = common.parent
    return str(common)

resolved_img_dir = NEW_IMAGES_DIR
if not resolved_img_dir or resolved_img_dir == "auto" or not Path(resolved_img_dir).exists():
    detected = _find_kaggle_image_root()
    if not detected:
        raise FileNotFoundError("Không tìm thấy ảnh mới trong /kaggle/input! Kiểm tra lại bạn đã Add Input dataset mới chưa.")
    resolved_img_dir = detected
print("Using images dir:", resolved_img_dir)

ip = get_ipython()
cfg_flag = f"--config configs/pipeline_kaggle.yaml --work-dir {shlex.quote(WORK_DIR)}"
img_flag = f"--images-dir {shlex.quote(resolved_img_dir)}"
lbl_flag = f"--labels-dir {shlex.quote(NEW_LABELS_OR_DATA)}" if NEW_LABELS_OR_DATA else "--labels-dir ''"

if MODE == "prelabel":
    cmd = f"python -u run_pipeline.py warmup-export {cfg_flag} {img_flag} {lbl_flag} --num-images {NUM_IMAGES} --batch-name {shlex.quote(BATCH_NAME)}"
    print("Running:", cmd)
    ip.system(cmd)
elif MODE == "train_cvat":
    if not NEW_LABELS_OR_DATA:
        raise ValueError("MODE='train_cvat' cần điền NEW_LABELS_OR_DATA tới folder labels/ hoặc file .zip/.json CVAT!")
    cmd_imp = f"python -u run_pipeline.py import-cvat {cfg_flag} {img_flag} --source {shlex.quote(NEW_LABELS_OR_DATA)}"
    cmd_trn = f"python -u run_pipeline.py train {cfg_flag} {img_flag} --run-name {shlex.quote(RUN_NAME)} --weights {shlex.quote(WEIGHTS_INIT)} --epochs {EPOCHS} --batch {BATCH} --imgsz {IMGSZ}"
    print("Running:", cmd_imp)
    ip.system(cmd_imp)
    print("Running:", cmd_trn)
    ip.system(cmd_trn)
elif MODE == "train_yaml":
    if not NEW_LABELS_OR_DATA:
        raise ValueError("MODE='train_yaml' cần điền NEW_LABELS_OR_DATA tới file data.yaml!")
    cmd_trn = f"python -u run_pipeline.py train {cfg_flag} --data {shlex.quote(NEW_LABELS_OR_DATA)} --run-name {shlex.quote(RUN_NAME)} --weights {shlex.quote(WEIGHTS_INIT)} --epochs {EPOCHS} --batch {BATCH} --imgsz {IMGSZ}"
    print("Running:", cmd_trn)
    ip.system(cmd_trn)
elif MODE == "demo_50":
    n = NUM_IMAGES if NUM_IMAGES > 0 else 50
    cmd = f"python -u run_pipeline.py run-demo-50 {cfg_flag} {img_flag} {lbl_flag} --num-images {n} --epochs {EPOCHS}"
    print("Running:", cmd)
    ip.system(cmd)
else:
    raise ValueError(f"MODE không hợp lệ: {MODE}")

## Bước 2 (Tùy chọn): Sau khi sửa nhãn CVAT, import lại & Train YOLO11m-seg

In [ ]:
# Sau khi đưa {BATCH_NAME}_coco.zip lên CVAT sửa và upload lại file zip đã sửa vào /kaggle/input/...
# !python -u run_pipeline.py import-cvat --config configs/pipeline_kaggle.yaml --work-dir {WORK_DIR} --images-dir {resolved_img_dir} --source /kaggle/input/cvat-reviewed/reviewed.zip
# !python -u run_pipeline.py train --config configs/pipeline_kaggle.yaml --work-dir {WORK_DIR} --images-dir {resolved_img_dir} --run-name {RUN_NAME} --epochs {EPOCHS} --batch {BATCH} --imgsz {IMGSZ}

## Bước 3: Vẽ ảnh xem trước & Nén tải kết quả

In [ ]:
import cv2
from pathlib import Path
from tqdm.auto import tqdm
from IPython.display import display, Image
from src.config import load_taxonomy
from src.geometry import read_yolo_seg_file
from src.visualize import draw_instances_overlay

tax = load_taxonomy("configs/classes_11.yaml")
img_dir = Path(WORK_DIR) / "cvat_export" / BATCH_NAME / "yolo_seg" / "images"
lbl_dir = Path(WORK_DIR) / "cvat_export" / BATCH_NAME / "yolo_seg" / "labels"
out_dir = Path(WORK_DIR) / "visuals_preview"

if img_dir.exists() and lbl_dir.exists():
    out_dir.mkdir(parents=True, exist_ok=True)
    imgs = sorted(img_dir.glob("*"))[:30]
    for p in tqdm(imgs, desc="Render preview", unit="img"):
        im = cv2.imread(str(p))
        if im is None:
            continue
        h, w = im.shape[:2]
        insts = read_yolo_seg_file(lbl_dir / f"{p.stem}.txt", w, h)
        draw_instances_overlay(p, insts, tax, out_dir / p.name)
    for sample in sorted(out_dir.glob("*"))[:4]:
        print(sample.name)
        display(Image(filename=str(sample), width=800))
else:
    print("Chưa thấy thư mục output prelabel.")

In [ ]:
%cd /kaggle/working/hackathon_vinai
!zip -rq /kaggle/working/ket_qua_new_dataset.zip {WORK_DIR}/runs {WORK_DIR}/cvat_export {WORK_DIR}/auto_labeled {WORK_DIR}/reports {WORK_DIR}/visuals_preview 2>/dev/null || true
!ls -lh /kaggle/working/ket_qua_new_dataset.zip
%cd /kaggle/working
from IPython.display import FileLink
FileLink('ket_qua_new_dataset.zip')